**Transform Sprints Data**

1. Read bronze sprints table
2. Keep only the columns required for analytics (Drop url column)
3. Standardise column names using snake_case-(constructorId constructor_id, driverId driver_id, raceName → race_name, positionText finish_position_text)
4. Rename columns to make them more meaningful (date race_date, grid grid_position, laps completed_laps, number →car_number, position finish_position)
5. Filter out rows where season, round, custructor_id or driver_id is null (business key validation)
6. Remove duplicate records
7. Transform values of column race_name to Title Case
8. Write the transformed data to silver sprints table

In [0]:
%run "/Workspace/Users/ganeshgadade157@gmail.com/Projects/Formula-f1/00.common/01.envioment_config"

In [0]:
from pyspark.sql import functions as f

folder_name = 'sprints'
file_name   = 'sprints.parquet'
silver_table = f"dev.silver.{folder_name}"

# 1. Read Bronze
raw_df = spark.read.format('parquet')\
              .load(f"{bronze_path}/{folder_name}/{file_name}")

# 2. Cast types + Drop url
raw_df = raw_df.withColumn('date', f.col('date').cast('date'))\
               .withColumn('round', f.col('round').cast('integer'))\
               .withColumn('season',f.col('season').cast('integer'))\
               .withColumn('grid', f.col('grid').cast('integer'))\
               .withColumn('laps', f.col('laps').cast('integer'))\
               .withColumn('number',f.col('number').cast('integer'))\
               .withColumn('points', f.col('points').cast('integer'))\
               .withColumn('position', f.col('position').cast('integer'))\
               .withColumn('positionText',f.col('positionText').cast('integer'))\
               .withColumn('ingestion_timestamp', f.col('ingestion_timestamp').cast('timestamp'))\
               .drop('url')                       

# 3. Rename columns
raw_df = raw_df.withColumnRenamed('constructorId','constructor_id')\
               .withColumnRenamed('driverId', 'driver_id')\
               .withColumnRenamed('raceName',  'race_name')\
               .withColumnRenamed('positionText', 'finish_position_text')\
               .withColumnRenamed('date','race_date')\
               .withColumnRenamed('grid','grid_position')\
               .withColumnRenamed('laps', 'completed_laps')\
               .withColumnRenamed('number',   'car_number')\
               .withColumnRenamed('position', 'finish_position')

# 4. Filter null business key
raw_df = raw_df.filter(
    f.col('season').isNotNull() &
    f.col('round').isNotNull() &
    f.col('constructor_id').isNotNull() &
    f.col('driver_id').isNotNull()
)

# 5. Remove duplicates
raw_df = raw_df.dropDuplicates(['season', 'round', 'constructor_id', 'driver_id'])    

# 6. Title Case
raw_df = raw_df.withColumn('race_name', f.initcap(f.col('race_name')))

#7. Add serial number 
raw_df = raw_df.withColumn('sprint_result_id', f.monotonically_increasing_id() + 1)

# 8. Write to Silver
raw_df.write\
      .format('delta')\
      .mode('overwrite')\
      .saveAsTable(silver_table)